# Experiment 1: Implementation of Simple Linear Regression

### Mathematical Model
$$\hat{y} = b_0 + b_1 x$$
- **Slope ($b_1$):** $\frac{\sum (x_i - \bar{x})(y_i - \bar{y})}{\sum (x_i - \bar{x})^2}$
- **Intercept ($b_0$):** $\bar{y} - b_1 \bar{x}$
- **Loss Function (MSE):** $J = \frac{1}{n} \sum (y_i - \hat{y}_i)^2$

### Step 1: Import Required Libraries

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

### Step 2: Create / Load & Explore the Dataset (Self-Contained)

In [ ]:

np.random.seed(42)
years_exp = np.round(np.linspace(1.0, 15.0, 50) + np.random.normal(0, 0.3, 50), 2)
years_exp = np.clip(years_exp, 0.5, 16.0)
monthly_salary = np.round(25000 + 3200 * years_exp + np.random.normal(0, 2800, 50), 2)

df = pd.DataFrame({
    'YearsExperience': years_exp,
    'MonthlySalary': monthly_salary
})

print("Dataset Shape:", df.shape)
print("\nMissing Values:\n", df.isnull().sum())
display(df.head())
display(df.describe())

### Step 3: Exploratory Scatter Plot & Train-Test Split

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(df['YearsExperience'], df['MonthlySalary'], color='#4F46E5', alpha=0.8, edgecolor='#1E1B4B')
plt.title('Years of Experience vs. Monthly Salary (Raw Data)', fontsize=13, fontweight='bold')
plt.xlabel('Years of Experience')
plt.ylabel('Monthly Salary (INR)')
plt.grid(True, linestyle='--', alpha=0.4)
plt.show()

X = df[['YearsExperience']].values
y = df['MonthlySalary'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Training Samples: {len(X_train)} | Testing Samples: {len(X_test)}")

### Step 4: Implement OLS From Scratch & Train `LinearRegression` Model

In [ ]:
# Method A: From-Scratch Closed-Form OLS
x_tr_1d = X_train.ravel()
x_mean, y_mean = np.mean(x_tr_1d), np.mean(y_train)
b1_scratch = np.sum((x_tr_1d - x_mean) * (y_train - y_mean)) / np.sum((x_tr_1d - x_mean) ** 2)
b0_scratch = y_mean - b1_scratch * x_mean
print(f"From-Scratch OLS -> Intercept (b0): {b0_scratch:.2f}, Slope (b1): {b1_scratch:.2f}")

# Method B: scikit-learn LinearRegression
model = LinearRegression()
model.fit(X_train, y_train)
print(f"Scikit-Learn OLS -> Intercept (b0): {model.intercept_:.2f}, Slope (b1): {model.coef_[0]:.2f}")

### Step 5: Predictions & Model Evaluation

In [ ]:
y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print(f"MAE  : {mae:.2f}")
print(f"MSE  : {mse:.2f}")
print(f"RMSE : {rmse:.2f}")
print(f"R2   : {r2:.4f}")

# Comparison DataFrame
results_df = pd.DataFrame({'Actual': y_test, 'Predicted': np.round(y_pred, 2), 'Residual': np.round(y_test - y_pred, 2)})
display(results_df.head(10))

### Step 6: Visualize Fitted Regression Line & Residual Plot

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# Regression Line
ax1.scatter(X_train, y_train, color='#4F46E5', label='Training Data', alpha=0.75)
ax1.scatter(X_test, y_test, color='#0D9488', marker='s', label='Test Data', alpha=0.85)
x_line = np.linspace(X.min(), X.max(), 100).reshape(-1, 1)
ax1.plot(x_line, model.predict(x_line), color='#E11D48', lw=2.5, label='Fitted Regression Line')
ax1.set_title('Simple Linear Regression Fit', fontweight='bold')
ax1.set_xlabel('Years of Experience')
ax1.set_ylabel('Monthly Salary')
ax1.legend()
ax1.grid(True, linestyle='--', alpha=0.4)

# Residual Plot
residuals = y_test - y_pred
ax2.scatter(y_pred, residuals, color='#D97706', edgecolor='#1E1B4B', s=70)
ax2.axhline(0, color='#E11D48', linestyle='--', lw=2)
ax2.set_title('Residual Plot (Test Set)', fontweight='bold')
ax2.set_xlabel('Predicted Salary')
ax2.set_ylabel('Residuals (Actual - Predicted)')
ax2.grid(True, linestyle='--', alpha=0.4)

plt.tight_layout()
plt.show()

### Step 7: Predict for New Unseen Input

In [ ]:
new_exp = np.array([[6.5], [10.0]])
preds = model.predict(new_exp)
for exp, sal in zip(new_exp.ravel(), preds):
    print(f"Predicted Monthly Salary for {exp} years experience = {sal:,.2f}")